**COMANDO WSL2 — EXECUTAR FASE COMPLETA**

```bash
make fase3
```

**TÍTULO**

F3N1 — Organização e normalização da série histórica semanal (1981–2026)

**CONTEXTO**

A fase3_nino investiga a evolução do El Niño estritamente no Pacífico (região Niño 3.4), da gênese ao decaimento. A base de entrada é a matriz semanal W-SUN publicada pela Fase 2 a partir de NOAA OISST v2.1 (SSTA), ERA5 (atmosfera) e UFS+GLORYS (oceano subsuperficial). Comparações entre variáveis de unidades e escalas distintas exigem uma base comum de anomalias normalizadas.

**DESAFIO**

Hipótese: após remoção da sazonalidade e padronização Z-score, as anomalias das variáveis oceânicas e atmosféricas tornam-se diretamente comparáveis, revelando a coevolução do sistema acoplado. Objetivos: (i) estruturar as séries semanais 1981–2026; (ii) remover o ciclo sazonal de todas as variáveis; (iii) padronizar via Z-score com período climatológico de referência 1991–2020.

**METODOLOGIA**

Para cada variável x(t): anomalia semanal x'(t) = x(t) − c(w(t)), onde c é a climatologia por semana-do-ano (1..53) ajustada em 1991–2020 e suavizada circularmente (janela de 5 semanas); em seguida z(t) = (x'(t) − μ)/σ com μ e σ do período de referência. Colunas que já são anomalias na F2 (sufixo _anom e nino34_ssta) recebem apenas o Z-score. Não há preenchimento de lacunas: ausências permanecem explícitas.

**RESULTADOS ESPERADOS**

TabF3N1_zscores_semanais.csv (matriz completa normalizada), TabF3N1_resumo_estatistico.csv e FigF3N1_series_normalizadas (png+pdf, 600 dpi) com painéis oceânico e atmosférico das anomalias padronizadas.

**REFERÊNCIAS BIBLIOGRÁFICAS**

1. HUANG, B. et al. Improvements of the Daily Optimum Interpolation Sea Surface Temperature (DOISST) Version 2.1. Journal of Climate, v. 34, p. 2923-2939, 2021. DOI: https://doi.org/10.1175/JCLI-D-20-0166.1
2. HERSBACH, H. et al. The ERA5 global reanalysis. Quarterly Journal of the Royal Meteorological Society, v. 146, p. 1999-2049, 2020. DOI: https://doi.org/10.1002/qj.3803
3. WILKS, D. S. Statistical Methods in the Atmospheric Sciences. 3. ed. Academic Press, 2011. DOI: https://doi.org/10.1016/C2010-0-66249-4

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))
from nino_brasil import fase3_nino as f3

f3.ensure_dirs()
master = f3.load_master_weekly()
fisicas = [c for c in master.columns if c != 'ocean_source_code']
print(f'Master semanal F2: {master.shape[0]} semanas x {len(fisicas)} variaveis fisicas'
      f" | simulado={master.attrs.get('dado_simulado', False)}")
print(f'Periodo: {master.index.min().date()} a {master.index.max().date()}')

Master semanal F2: 2376 semanas x 44 variaveis fisicas | simulado=False
Periodo: 1981-01-04 a 2026-07-12


In [2]:
zscores = f3.deseason_and_zscore(master[fisicas])
f3.save_table(zscores, 'TabF3N1_zscores_semanais')

resumo = pd.DataFrame({
    'media': zscores.mean(), 'desvio': zscores.std(ddof=1),
    'minimo': zscores.min(), 'maximo': zscores.max(),
    'semanas_validas': zscores.notna().sum(),
    'cobertura_pct': (100.0 * zscores.notna().mean()).round(2),
}).round(3)
f3.save_table(resumo, 'TabF3N1_resumo_estatistico')
resumo.head(12)

[tabela] TabF3N1_zscores_semanais.csv (2376x44) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3
[tabela] TabF3N1_resumo_estatistico.csv (44x6) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


,media,desvio,minimo,maximo,semanas_validas,cobertura_pct
d20_m,-0.015,1.034,-3.435,2.742,2340,98.48
div200_anom,0.032,0.982,-2.731,6.536,2374,99.92
div500_anom,0.038,1.005,-4.436,3.475,2374,99.92
div850_anom,0.046,0.995,-5.708,3.014,2374,99.92
mslp_anom,0.033,1.029,-3.774,3.313,2374,99.92
nino34_ssta,-0.063,1.040,-3.052,3.548,2340,98.48
ohc_0_100,-0.084,1.033,-3.122,2.760,2340,98.48
ohc_0_300,-0.047,0.995,-2.933,2.738,2340,98.48
ohc_0_700,-0.067,0.994,-2.927,2.834,2340,98.48
ohc_300_700,-0.142,1.116,-3.693,2.913,2340,98.48


In [3]:
oceanicas = [c for c in ['nino34_ssta','d20_m','ohc_0_300','wwv','ssh_m','t100m'] if c in zscores]
atmosfericas = [c for c in ['tau_x_anom','u10_anom','u850_anom','mslp_anom','tcwv_anom','ssr_anom'] if c in zscores]
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for coluna in oceanicas:
    axes[0].plot(zscores.index, zscores[coluna], lw=0.7, label=coluna)
for coluna in atmosfericas:
    axes[1].plot(zscores.index, zscores[coluna], lw=0.7, label=coluna)
axes[0].set_title('Anomalias normalizadas (Z-score) — bloco oceanico')
axes[1].set_title('Anomalias normalizadas (Z-score) — bloco atmosferico')
for ax in axes:
    ax.axhline(0, color='k', lw=0.5)
    ax.legend(loc='upper left', ncol=3, fontsize=7)
    ax.set_ylabel('z')
fig.tight_layout()
f3.save_table(zscores[oceanicas + atmosfericas], 'FigF3N1_series_normalizadas_dados')
f3.save_figure(fig, 'FigF3N1_series_normalizadas')
plt.close(fig)

[tabela] FigF3N1_series_normalizadas_dados.csv (2376x12) -> C:\DEV\NINO26\data\processed\numeric-tables\fase3


[figura] FigF3N1_series_normalizadas.png (+pdf) -> C:\DEV\NINO26\data\processed\figures\fase3
